In [4]:
import polars as pl
import os


# --- 1. DEFINE MATCHING FILES ---
# Example: Cow 1 on July 25th
LABEL_FILE = '../data/sensor_data/behavior_labels/individual/C01_0725.csv'
IMMU_FILE = '../data/sensor_data/main_data/immu/T01/T01_0725.csv' # note: matching dates matter.
THI_FILE = '../data/sensor_data/main_data/thi/S01.csv'

print("1. Loading and Standardizing Timestamps...")


1. Loading and Standardizing Timestamps...


In [5]:

# --- 2. LOAD AND CLEAN BRONZE DATA ---

# 2A. Process Labels
# Convert integer seconds to standard Polars Datetime
df_labels = (
    pl.read_csv(LABEL_FILE)
    .with_columns(
        pl.from_epoch("timestamp", time_unit="s").alias("datetime_utc")
    )
    # We must sort by time for an As-Of Join to work!
    .sort("datetime_utc")
    .select(["datetime_utc", "behavior"])
)
#df_labels

# 2B. Process IMMU (Collar Physics)
# Convert float seconds to microseconds, then to Polars Datetime
df_immu = (
    pl.read_csv(IMMU_FILE)
    .with_columns(
        pl.from_epoch((pl.col("timestamp") * 1e6).cast(pl.Int64), time_unit="us").alias("datetime_utc")
    )
    .sort("datetime_utc")
    .select(["datetime_utc", "accel_x_mps2", "accel_y_mps2", "accel_z_mps2"])
)

# 2C. Process THI (Weather)
df_thi = (
    pl.read_csv(THI_FILE)
    .with_columns(
        pl.from_epoch((pl.col("timestamp") * 1e6).cast(pl.Int64), time_unit="us").alias("datetime_utc")
    )
    .sort("datetime_utc")
    .select(["datetime_utc", "temperature_C", "THI"])
)

print("2. Executing As-Of Joins (Creating Silver Layer)...")



2. Executing As-Of Joins (Creating Silver Layer)...
Silver Layer complete! Shape: (864000, 7)


In [9]:
# --- 3. CREATE THE SILVER LAYER (The Merge) ---
# We use the high-frequency IMMU data as our base.
# note: strategy="backward" means "get the most recent label/weather reading that happened before this millisecond"
df_silver = (
    df_immu
    .join_asof(df_labels, on="datetime_utc", strategy="backward")
    .join_asof(df_thi, on="datetime_utc", strategy="backward")
)

# Drop rows where weather or labels are completely missing => easy solution?
df_silver = df_silver.drop_nulls()


print(f"Silver Layer complete! Shape: {df_silver.shape}")

Silver Layer complete! Shape: (864000, 7)


In [6]:


print("3. Aggregating to Hourly Metrics (Creating Gold Layer)...")

# --- 4. CREATE THE GOLD LAYER (The Business Logic) ---
# We compress millions of rows down into 24 clean hourly rows
df_gold = (
    df_silver
    # Truncate the exact millisecond down to the flat Hour (e.g., 14:00:00)
    .with_columns(pl.col("datetime_utc").dt.truncate("1h").alias("hour"))
    .group_by("hour")
    .agg([
        # 1. Physics: Standard Deviation of acceleration = How much is the cow moving?
        pl.col("accel_x_mps2").std().alias("activity_variance_x"),
        pl.col("accel_y_mps2").std().alias("activity_variance_y"),
        
        # 2. Weather: Average Temp and THI for that hour
        pl.col("temperature_C").mean().alias("avg_temp_C"),
        pl.col("THI").mean().alias("avg_THI"),
        
        # 3. Behavior: Count how many sensor ticks were spent in Behavior '0', '1', etc.
        # (Assuming behavior '0' might be resting, '1' eating, etc. Check the dataset docs!)
        (pl.col("behavior") == 0).sum().alias("behavior_0_ticks"),
        (pl.col("behavior") == 1).sum().alias("behavior_1_ticks"),
    ])
    .sort("hour")
)
 


3. Aggregating to Hourly Metrics (Creating Gold Layer)...


In [7]:
# --- 5. SAVE THE FINAL PIPELINE OUTPUT ---
os.makedirs('../data/gold/', exist_ok=True)
df_gold.write_parquet('../data/gold/cow01_daily_summary.parquet')

print("✅ Pipeline Complete! Gold Layer saved.")
print("\n--- FIRST 5 HOURS OF GOLD DATA ---")
print(df_gold.head())

✅ Pipeline Complete! Gold Layer saved.

--- FIRST 5 HOURS OF GOLD DATA ---
shape: (5, 7)
┌──────────────┬──────────────┬──────────────┬────────────┬───────────┬──────────────┬─────────────┐
│ hour         ┆ activity_var ┆ activity_var ┆ avg_temp_C ┆ avg_THI   ┆ behavior_0_t ┆ behavior_1_ │
│ ---          ┆ iance_x      ┆ iance_y      ┆ ---        ┆ ---       ┆ icks         ┆ ticks       │
│ datetime[μs] ┆ ---          ┆ ---          ┆ f64        ┆ f64       ┆ ---          ┆ ---         │
│              ┆ str          ┆ str          ┆            ┆           ┆ u32          ┆ u32         │
╞══════════════╪══════════════╪══════════════╪════════════╪═══════════╪══════════════╪═════════════╡
│ 2023-07-25   ┆ null         ┆ null         ┆ 22.152467  ┆ 70.57635  ┆ 36000        ┆ 0           │
│ 05:00:00     ┆              ┆              ┆            ┆           ┆              ┆             │
│ 2023-07-25   ┆ null         ┆ null         ┆ 21.617983  ┆ 69.903883 ┆ 36000        ┆ 0           │
│ 

hour,activity_variance_x,activity_variance_y,avg_temp_C,avg_THI,behavior_0_ticks,behavior_1_ticks
datetime[μs],str,str,f64,f64,u32,u32
2023-07-25 05:00:00,null,null,22.152467,70.57635,36000,0
2023-07-25 06:00:00,null,null,21.617983,69.903883,36000,0
2023-07-25 07:00:00,null,null,21.18275,69.308467,34380,0
2023-07-25 08:00:00,null,null,20.953517,68.94605,0,1370
2023-07-25 09:00:00,null,null,21.1503,69.24265,17950,330
…,…,…,…,…,…,…
2023-07-26 00:00:00,null,null,27.801583,77.688083,0,100
2023-07-26 01:00:00,null,null,26.194383,75.957567,0,830
2023-07-26 02:00:00,null,null,24.7938,74.2736,0,650


# 